# חיזוי תחום ההתמחות של מפתחי ⁦GitHub⁩ לפי קשרי מעקב הדדיים ברשת

## 1. כותרת ושאלת מחקר
**שאלת המחקר:**

באיזו מידה מאפיינים מבניים של רשת קשרי המעקב ההדדיים בין מפתחי ⁦GitHub⁩, ובפרט שכנות, מרכזיות וקהילות, מאפשרים לחזות האם מפתח משתייך לתחום ⁦Web⁩ או ⁦Machine Learning⁩?

המחברת מציגה תהליך מלא של בניית גרף, ניתוח מבני, מודלי חיזוי מבוססי רשת בלבד, אלגוריתם ⁦Label Propagation⁩, שיפור בעזרת קהילות, כיול סף החלטה וניתוח שגיאות.

<style>
/* RTL notebook text with isolated LTR terms. */
.jp-RenderedMarkdown,
.text_cell_render {
  direction: rtl;
  text-align: right;
  unicode-bidi: plaintext;
  line-height: 1.65;
}
.jp-RenderedMarkdown h1,
.jp-RenderedMarkdown h2,
.jp-RenderedMarkdown h3,
.text_cell_render h1,
.text_cell_render h2,
.text_cell_render h3 {
  direction: rtl;
  text-align: right;
}
.jp-RenderedMarkdown ul,
.jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol {
  direction: rtl;
  text-align: right;
}
.jp-RenderedMarkdown code,
.jp-RenderedMarkdown pre,
.jp-RenderedMarkdown table,
.text_cell_render code,
.text_cell_render pre,
.text_cell_render table {
  direction: ltr;
  text-align: left;
  unicode-bidi: isolate;
}
.jp-CodeCell,
.jp-InputArea,
.jp-OutputArea {
  direction: ltr;
  text-align: left;
}
</style>

## 2. מבוא ומוטיבציה
⁦GitHub⁩ היא פלטפורמה חברתית-מקצועית שבה קשרי מעקב משקפים לעיתים תחומי עניין מקצועיים. לכן ניתן לייצג את הבעיה כגרף:
- **צומת (⁦Node⁩):** מפתח ⁦GitHub⁩.
- **קשת (⁦Edge⁩):** קשר מעקב הדדי בין שני מפתחים.

הפרויקט מתמקד בגישת **⁦Network-Only⁩**: החיזוי מתבסס על מבנה הרשת וקשרי המעקב ההדדיים בלבד, ללא שימוש בתוכן טקסטואלי של פרופילים או מאגרים.

הניסוי מוגדר כ-**⁦Transductive Node Classification⁩**: מבנה הגרף ידוע מראש עבור כל הצמתים, כולל צמתי הבדיקה, אבל תוויות ה-⁦test⁩ אינן משמשות באימון או בבחירת פרמטרים. המודל רשאי להשתמש בקשרים ברשת, אך לא בתשובות האמיתיות של קבוצת הבדיקה. הגדרה זו מתאימה לבעיה משום שבמערכת חברתית כמו ⁦GitHub⁩ לעיתים מבנה הקשרים ידוע גם עבור משתמשים שתחום ההתמחות שלהם עדיין אינו ידוע.

בהתאם לכלל המתודולוגי של העבודה, הקובץ `musae_git_features.json` אינו משמש כקלט למודל החיזוי.

## 3. תיאור הדאטה
הדאטה מבוסס על ⁦MUSAE GitHub Social Network⁩ עם תיוג בינארי:
- `0` = ⁦Web⁩
- `1` = ⁦Machine Learning⁩

סטטיסטיקה מרכזית:
- צמתים: 37,700
- קשתות: 289,003
- צפיפות: 0.000407
- דרגה ממוצעת: 15.332
- רכיבים קשירים: 1
- ⁦Web⁩: 27,961 (74.17%)
- ⁦ML⁩: 9,739 (25.83%)

המחלקות אינן מאוזנות, ולכן ⁦Accuracy⁩ לבדו אינו מדד מספק.

### טעינת הנתונים והכנה לעיבוד
בשלב זה נטענים הקבצים, נבנים מבני הנתונים הבסיסיים (⁦DataFrame⁩ ומילון תוויות), ומוגדרות תיקיות הפלט לשמירת טבלאות וגרפים.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Figures dir: {FIGURES_DIR}")
print(f"Tables dir: {TABLES_DIR}")

Project root: C:\Dev\Graph-Algo-Course\github-social-network
Data dir: C:\Dev\Graph-Algo-Course\github-social-network\data
Figures dir: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures
Tables dir: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from IPython.display import display

from data_loader import load_dataset
from graph_analysis import (
    compute_basic_stats,
    compute_label_distribution,
    compute_edge_type_counts,
    compute_homophily,
    compute_label_assortativity,
)
from visualization import (
    plot_label_distribution,
    plot_degree_distribution,
    plot_edge_type_distribution,
    plot_sampled_network,
    plot_community_purity_error_rates,
    plot_error_type_counts,
)
from modeling import (
    prepare_train_test_split,
    majority_class_baseline,
    neighbor_majority_classifier,
    label_propagation_classifier,
    extract_graph_neighbor_features,
    train_logistic_regression_graph_features,
    predict_logistic_regression_graph_features,
    evaluate_predictions,
    run_repeated_experiments,
    summarize_repeated_results,
    evaluate_by_degree_group,
    evaluate_errors_by_community_purity,
    summarize_error_types,
    evaluate_thresholds,
    select_best_threshold,
    run_repeated_threshold_tuning,
    summarize_threshold_tuning_results,
)
from community_analysis import (
    detect_louvain_communities,
    summarize_communities,
    add_community_features,
)

In [3]:
G, edges_df, labels_df, labels_dict = load_dataset(DATA_DIR)
print(f"Loaded edges: {len(edges_df):,}")
print(f"Loaded labeled nodes: {len(labels_df):,}")

Loaded edges: 289,003
Loaded labeled nodes: 37,700


## 4. בניית הגרף
בשלב זה נבנה גרף לא-מכוון של קשרי מעקב הדדיים.

הייצוג הגרפי מאפשר שימוש ישיר בכלים שנלמדו בקורס: דרגות, הומופיליה, קהילות, ניתוח יציבות וחיזוי על בסיס מבנה רשת.

In [4]:
print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")

Graph nodes: 37,700
Graph edges: 289,003


## 5. ניתוח ראשוני של הרשת
מטרת השלב היא להבין את מאפייני המבנה לפני חיזוי:
- גודל הרשת וקישוריות
- התפלגות דרגות
- פיזור התוויות

ניתוח זה מספק הקשר חשוב לפירוש ביצועי המודלים בהמשך.

In [5]:
basic_stats = compute_basic_stats(G, labels_df)
label_distribution_df = compute_label_distribution(labels_df)

print("Basic stats:")
for key, value in basic_stats.items():
    print(f"- {key}: {value}")

print()
print("Label distribution:")
display(label_distribution_df)

Basic stats:
- num_nodes: 37700
- num_edges: 289003
- density: 0.0004066878203117068
- average_degree: 15.331724137931035
- num_connected_components: 1
- largest_component_size: 37700
- largest_component_ratio: 1.0
- labeled_nodes_total: 37700
- labeled_nodes_in_graph: 37700
- labeled_node_ratio: 1.0

Label distribution:


,label,count,class_name,ratio
0,0,27961,Web,0.741671
1,1,9739,Machine Learning,0.258329


## 6. בדיקת הומופיליה
הומופיליה בודקת האם צמתים מאותה מחלקה נוטים להתחבר זה לזה.

תוצאות:
- web_web: 224,623
- ml_ml: 19,684
- web_ml: 44,696
- same-label edge rate: 0.845344

השיעור הגבוה מחזק את ההשערה שמבנה הקשרים מכיל אות חיזויי לתחום ההתמחות.

In [26]:
edge_type_counts = compute_edge_type_counts(G, labels_dict)
homophily_stats = compute_homophily(G, labels_dict)
label_assortativity_all = compute_label_assortativity(G, labels_dict)

print("Edge type counts:")
for key, value in edge_type_counts.items():
    print(f"- {key}: {value:,}")

print()
print("Homophily stats:")
for key, value in homophily_stats.items():
    print(f"- {key}: {value}")

print()
print(f"Label assortativity (exploratory, all labels): {label_assortativity_all:.6f}")
print("Note: This metric is used for descriptive analysis only and is not included in the model training.")

Edge type counts:
- web_web: 224,623
- ml_ml: 19,684
- web_ml: 44,696
- missing_label: 0

Homophily stats:
- homophily_rate: 0.8453441659775158
- same_label_edges: 244307
- different_label_edges: 44696
- labeled_edges: 289003

Label assortativity (exploratory, all labels): 0.377822
Note: This metric is used for descriptive analysis only and is not included in the model training.


In [7]:
label_plot_path = plot_label_distribution(label_distribution_df, FIGURES_DIR)
degree_plot_path = plot_degree_distribution(G, FIGURES_DIR)
edge_type_plot_path = plot_edge_type_distribution(edge_type_counts, FIGURES_DIR)

print("Saved figures:")
print(f"- {label_plot_path}")
print(f"- {degree_plot_path}")
print(f"- {edge_type_plot_path}")

C:\Dev\Graph-Algo-Course\github-social-network\src\visualization.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=label_distribution_df, x="class_name", y="count", palette="Set2", ax=ax)


Saved figures:
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\label_distribution.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\degree_distribution.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\edge_type_distribution.png


C:\Dev\Graph-Algo-Course\github-social-network\src\visualization.py:79: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=plot_df, x="edge_type", y="count", palette="Blues", ax=ax)


## 7. ניסוי חיזוי ראשון
נבדקים שלושה מודלים בסיסיים:
1. ⁦Majority Baseline⁩
2. ⁦Neighbor Majority⁩
3. ⁦Logistic Regression⁩ - ⁦Graph Features Only⁩

חשיבות ה-Baseline: הוא קו ייחוס מינימלי למדידת ערך של מודלים מתקדמים יותר.

בגלל חוסר האיזון בין המחלקות, ⁦Accuracy⁩ לבדו עלול להטעות. לכן נבחנים גם:
- ⁦Balanced Accuracy⁩
- ⁦Precision⁩
- ⁦Recall⁩
- ⁦F1⁩
- ⁦ROC-AUC⁩

In [8]:
train_nodes, test_nodes, y_train, y_test = prepare_train_test_split(
    labels_df,
    test_size=0.3,
    random_state=42,
)

default_label = int(pd.Series(y_train).mode().iloc[0])
train_label_assortativity = compute_label_assortativity(G, labels_dict, nodes=train_nodes)

print(f"Train size: {len(train_nodes):,}")
print(f"Test size: {len(test_nodes):,}")
print(f"Default label (majority in train): {default_label}")
print(f"Train-only label assortativity: {train_label_assortativity:.6f}")

Train size: 26,390
Test size: 11,310
Default label (majority in train): 0
Train-only label assortativity: 0.372837


In [9]:
# 1) Majority Baseline
y_pred_majority, y_score_majority = majority_class_baseline(y_train, y_test)
metrics_majority = evaluate_predictions(y_test, y_pred_majority, y_score_majority)

# 2) Neighbor Majority
y_pred_neighbor, y_score_neighbor = neighbor_majority_classifier(
    G,
    labels_dict,
    train_nodes,
    test_nodes,
    default_label=default_label,
)
metrics_neighbor = evaluate_predictions(y_test, y_pred_neighbor, y_score_neighbor)

# 3) Label Propagation - direct graph algorithm with train labels clamped
y_pred_label_prop, y_score_label_prop = label_propagation_classifier(
    G=G,
    labels_dict=labels_dict,
    train_nodes=train_nodes,
    test_nodes=test_nodes,
    default_label=default_label,
)
metrics_label_prop = evaluate_predictions(y_test, y_pred_label_prop, y_score_label_prop)

# 4) Logistic Regression - Graph Features Only
nodes_for_features = list(train_nodes) + list(test_nodes)
graph_features_df = extract_graph_neighbor_features(
    G,
    labels_dict,
    train_nodes=train_nodes,
    nodes=nodes_for_features,
).drop_duplicates(subset=["node_id"]).set_index("node_id")

X_train_graph = graph_features_df.reindex(train_nodes).reset_index()
X_test_graph = graph_features_df.reindex(test_nodes).reset_index()

logreg_graph_model, y_pred_logreg, y_score_logreg = train_logistic_regression_graph_features(
    X_train_graph,
    y_train,
    X_test_graph,
)
metrics_logreg = evaluate_predictions(y_test, y_pred_logreg, y_score_logreg)

for name, metrics in [
    ("Majority Baseline", metrics_majority),
    ("Neighbor Majority", metrics_neighbor),
    ("Label Propagation", metrics_label_prop),
    ("Logistic Regression - Graph Features Only", metrics_logreg),
]:
    print(name)
    print(
        f"  Accuracy={metrics['accuracy']:.4f} | Balanced Accuracy={metrics['balanced_accuracy']:.4f} | "
        f"Precision={metrics['precision']:.4f} | Recall={metrics['recall']:.4f} | "
        f"F1={metrics['f1']:.4f} | Macro F1={metrics['macro_f1']:.4f} | "
        f"Weighted F1={metrics['weighted_f1']:.4f} | ROC-AUC={metrics['roc_auc']:.4f}"
    )

Majority Baseline
  Accuracy=0.7416 | Balanced Accuracy=0.5000 | Precision=0.0000 | Recall=0.0000 | F1=0.0000 | Macro F1=0.4258 | Weighted F1=0.6316 | ROC-AUC=0.5000
Neighbor Majority
  Accuracy=0.8141 | Balanced Accuracy=0.6655 | Precision=0.8223 | Recall=0.3580 | F1=0.4988 | Macro F1=0.6924 | Weighted F1=0.7859 | ROC-AUC=0.7800
Label Propagation
  Accuracy=0.8262 | Balanced Accuracy=0.6885 | Precision=0.8409 | Recall=0.4035 | F1=0.5453 | Macro F1=0.7189 | Weighted F1=0.8028 | ROC-AUC=0.7896
Logistic Regression - Graph Features Only
  Accuracy=0.7991 | Balanced Accuracy=0.7536 | Precision=0.6014 | Recall=0.6595 | F1=0.6291 | Macro F1=0.7457 | Weighted F1=0.8020 | ROC-AUC=0.8396


In [10]:
def _row(model_name, m):
    return {
        "Model": model_name,
        "Accuracy": m["accuracy"],
        "Balanced Accuracy": m["balanced_accuracy"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "Macro F1": m["macro_f1"],
        "Weighted F1": m["weighted_f1"],
        "ROC-AUC": m["roc_auc"],
    }

model_results_df = pd.DataFrame([
    _row("Majority Baseline", metrics_majority),
    _row("Neighbor Majority", metrics_neighbor),
    _row("Label Propagation", metrics_label_prop),
    _row("Logistic Regression - Graph Features Only", metrics_logreg),
])
model_results_path = TABLES_DIR / "model_results.csv"
model_results_df.to_csv(model_results_path, index=False)
print(f"Saved: {model_results_path}")
display(model_results_df)

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\model_results.csv


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,Macro F1,Weighted F1,ROC-AUC
0,Majority Baseline,0.741645,0.500000,0.000000,0.000000,0.000000,0.425830,0.631629,0.500000
1,Neighbor Majority,0.814147,0.665515,0.822327,0.357974,0.498808,0.692365,0.785909,0.780000
2,Label Propagation,0.826172,0.688453,0.840942,0.403491,0.545328,0.718937,0.802840,0.789610
3,Logistic Regression - Graph Features Only,0.799116,0.753619,0.601436,0.659480,0.629122,0.745687,0.802022,0.839559


In [11]:
def save_confusion_matrix_plot(y_true, y_pred, title, output_path):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Pred Web (0)", "Pred ML (1)"],
        yticklabels=["True Web (0)", "True ML (1)"],
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Truth")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.close(fig)

cm_majority_path = FIGURES_DIR / "confusion_matrix_majority_baseline.png"
cm_neighbor_path = FIGURES_DIR / "confusion_matrix_neighbor_majority.png"
cm_label_prop_path = FIGURES_DIR / "confusion_matrix_label_propagation.png"
cm_logreg_path = FIGURES_DIR / "confusion_matrix_logistic_graph_features.png"

save_confusion_matrix_plot(y_test, y_pred_majority, "Majority Baseline", cm_majority_path)
save_confusion_matrix_plot(y_test, y_pred_neighbor, "Neighbor Majority", cm_neighbor_path)
save_confusion_matrix_plot(y_test, y_pred_label_prop, "Label Propagation", cm_label_prop_path)
save_confusion_matrix_plot(y_test, y_pred_logreg, "LogReg - Graph Features", cm_logreg_path)

print("Saved confusion matrix figures:")
print(f"- {cm_majority_path}")
print(f"- {cm_neighbor_path}")
print(f"- {cm_label_prop_path}")
print(f"- {cm_logreg_path}")

Saved confusion matrix figures:
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\confusion_matrix_majority_baseline.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\confusion_matrix_neighbor_majority.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\confusion_matrix_label_propagation.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\confusion_matrix_logistic_graph_features.png


## 8. בדיקת יציבות וניתוח שגיאות
כדי להקטין תלות בחלוקה אקראית בודדת, מריצים את הניסוי על 5 חלוקות ⁦Train/Test⁩ שונות ומדווחים ממוצע וסטיית תקן.

בנוסף מבוצע ניתוח שגיאות לפי קבוצות דרגה, כדי לבדוק האם ביצועי המודל משתנים בין צמתים עם מעט קשרים לבין צמתים עם קישוריות גבוהה.

In [12]:
repeated_results_df = run_repeated_experiments(
    G=G,
    labels_df=labels_df,
    n_runs=5,
    test_size=0.3,
    random_states=[42, 7, 21, 84, 123],
)

repeated_results_path = TABLES_DIR / "repeated_model_results.csv"
repeated_results_df.to_csv(repeated_results_path, index=False)

repeated_summary_df = summarize_repeated_results(repeated_results_df)
repeated_summary_path = TABLES_DIR / "repeated_model_results_summary.csv"
repeated_summary_df.to_csv(repeated_summary_path, index=False)

print(f"Saved: {repeated_results_path}")
print(f"Saved: {repeated_summary_path}")

display(repeated_results_df)
display(repeated_summary_df)

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\repeated_model_results.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\repeated_model_results_summary.csv


,run,random_state,model,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1,roc_auc
0,1,42,Majority Baseline,0.741645,0.500000,0.000000,0.000000,0.000000,0.425830,0.631629,0.500000
1,1,42,Neighbor Majority,0.814147,0.665515,0.822327,0.357974,0.498808,0.692365,0.785909,0.780000
2,1,42,Label Propagation,0.826172,0.688453,0.840942,0.403491,0.545328,0.718937,0.802840,0.789610
3,1,42,Logistic Regression - Graph Features Only,0.799116,0.753619,0.601436,0.659480,0.629122,0.745687,0.802022,0.839559
4,1,42,Logistic Regression - Graph + Community Features,0.814324,0.783720,0.621311,0.720397,0.667195,0.769220,0.818527,0.866639
5,2,7,Majority Baseline,0.741645,0.500000,0.000000,0.000000,0.000000,0.425830,0.631629,0.500000
6,2,7,Neighbor Majority,0.818391,0.676071,0.818649,0.381588,0.520542,0.704261,0.793050,0.789713
7,2,7,Label Propagation,0.828205,0.693280,0.839695,0.414100,0.554664,0.724120,0.806016,0.808894
8,2,7,Logistic Regression - Graph Features Only,0.805393,0.763315,0.611575,0.676249,0.642288,0.754314,0.808454,0.840449
9,2,7,Logistic Regression - Graph + Community Features,0.817153,0.792764,0.622560,0.742300,0.677178,0.774817,0.822005,0.872493


,Model,Accuracy Mean,Accuracy Std,Balanced Accuracy Mean,Balanced Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,Macro F1 Mean,Macro F1 Std,Weighted F1 Mean,Weighted F1 Std,ROC-AUC Mean,ROC-AUC Std
0,Label Propagation,0.828470,0.001761,0.691630,0.002203,0.849505,0.010077,0.408487,0.004032,0.551671,0.004387,0.722809,0.002686,0.805518,0.001894,0.802059,0.009291
1,Logistic Regression - Graph + Community Features,0.816127,0.003347,0.786274,0.004992,0.624225,0.006237,0.724504,0.010672,0.670602,0.006292,0.771534,0.004177,0.820313,0.003242,0.868779,0.005186
2,Logistic Regression - Graph Features Only,0.803307,0.003577,0.758942,0.005337,0.608909,0.005998,0.667146,0.009236,0.636693,0.007292,0.750921,0.004812,0.806126,0.003623,0.840940,0.003707
3,Majority Baseline,0.741645,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.425830,0.000000,0.631629,0.000000,0.500000,0.000000
4,Neighbor Majority,0.818780,0.003222,0.674080,0.005216,0.831198,0.012751,0.374675,0.009899,0.516468,0.010741,0.702481,0.006256,0.792379,0.004110,0.785099,0.005350


In [13]:
# Repeated-run plots (current pipeline)
fig1, ax1 = plt.subplots(figsize=(9, 5))
sns.barplot(data=repeated_summary_df, x="Model", y="F1 Mean", palette="Set2", ax=ax1)
ax1.set_title("Repeated Runs: Mean F1 by Model")
ax1.set_xlabel("Model")
ax1.set_ylabel("Mean F1")
ax1.tick_params(axis="x", rotation=20)
fig1.tight_layout()
repeated_f1_path = FIGURES_DIR / "repeated_f1_comparison.png"
fig1.savefig(repeated_f1_path, dpi=300)
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(9, 5))
sns.barplot(data=repeated_summary_df, x="Model", y="ROC-AUC Mean", palette="Set3", ax=ax2)
ax2.set_title("Repeated Runs: Mean ROC-AUC by Model")
ax2.set_xlabel("Model")
ax2.set_ylabel("Mean ROC-AUC")
ax2.tick_params(axis="x", rotation=20)
fig2.tight_layout()
repeated_roc_path = FIGURES_DIR / "repeated_roc_auc_comparison.png"
fig2.savefig(repeated_roc_path, dpi=300)
plt.close(fig2)

macro_weighted_plot_df = repeated_summary_df.melt(
    id_vars=["Model"],
    value_vars=["Macro F1 Mean", "Weighted F1 Mean"],
    var_name="Metric",
    value_name="Score",
)
fig_mw, ax_mw = plt.subplots(figsize=(9, 5))
sns.barplot(data=macro_weighted_plot_df, x="Model", y="Score", hue="Metric", palette="Set1", ax=ax_mw)
ax_mw.set_title("Repeated Runs: Macro F1 and Weighted F1")
ax_mw.set_xlabel("Model")
ax_mw.set_ylabel("Score")
ax_mw.tick_params(axis="x", rotation=20)
ax_mw.legend(title="Metric")
fig_mw.tight_layout()
repeated_macro_weighted_path = FIGURES_DIR / "repeated_macro_weighted_f1_comparison.png"
fig_mw.savefig(repeated_macro_weighted_path, dpi=300)
plt.close(fig_mw)

print(f"Saved: {repeated_f1_path}")
print(f"Saved: {repeated_roc_path}")
print(f"Saved: {repeated_macro_weighted_path}")

C:\Users\ofir8\AppData\Local\Temp\ipykernel_34344\2972200843.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=repeated_summary_df, x="Model", y="F1 Mean", palette="Set2", ax=ax1)
C:\Users\ofir8\AppData\Local\Temp\ipykernel_34344\2972200843.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=repeated_summary_df, x="Model", y="ROC-AUC Mean", palette="Set3", ax=ax2)


Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_f1_comparison.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_roc_auc_comparison.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_macro_weighted_f1_comparison.png


In [14]:
# Degree-based error analysis for LogReg graph-only from section 7
error_by_degree_df = evaluate_by_degree_group(
    G=G,
    test_nodes=test_nodes,
    y_true=y_test,
    y_pred=y_pred_logreg,
)
error_by_degree_path = TABLES_DIR / "error_analysis_by_degree_group.csv"
error_by_degree_df.to_csv(error_by_degree_path, index=False)

plot_df = error_by_degree_df.melt(
    id_vars=["degree_group", "count"],
    value_vars=["accuracy", "balanced_accuracy", "f1"],
    var_name="metric",
    value_name="value",
)

fig3, ax3 = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_df, x="degree_group", y="value", hue="metric", ax=ax3)
ax3.set_title("Error Analysis by Degree Group (LogReg Graph-Only)")
ax3.set_xlabel("Degree Group")
ax3.set_ylabel("Score")
ax3.tick_params(axis="x", rotation=15)
fig3.tight_layout()
error_degree_plot_path = FIGURES_DIR / "error_analysis_by_degree_group.png"
fig3.savefig(error_degree_plot_path, dpi=300)
plt.close(fig3)

print(f"Saved: {error_by_degree_path}")
print(f"Saved: {error_degree_plot_path}")
display(error_by_degree_df)

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\error_analysis_by_degree_group.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\error_analysis_by_degree_group.png


,degree_group,count,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1
0,degree_0_2,2877,0.694126,0.669656,0.558601,0.588645,0.573230,0.667438,0.695894
1,degree_3_5,2784,0.784483,0.760943,0.617773,0.703659,0.657925,0.750305,0.788266
2,degree_6_10,2207,0.824196,0.791020,0.628931,0.724638,0.673401,0.776564,0.828122
3,degree_11_30,2367,0.869877,0.783764,0.642202,0.648148,0.645161,0.782746,0.870110
4,degree_31_plus,1075,0.910698,0.814753,0.564286,0.692982,0.622047,0.785707,0.914656


## 9. ניתוח קהילות ⁦Louvain⁩
אלגוריתם ⁦Louvain⁩ מזהה קהילות צפופות וממקסם מודולריות, ולכן הוא רלוונטי ישירות לנושאי הקורס בניתוח גרפים וקהילות.

תוצאות עיקריות:
- מספר קהילות: 31
- מודולריות: 0.4534
- הקהילה הגדולה ביותר: גודל 8732, כ-93.5% ⁦Web⁩
- הקהילה השנייה בגודלה: גודל 7664, כ-71.5% ⁦ML⁩

המבנה הקהילתי שנמצא מצביע על סיגנל גלובלי משמעותי ברשת.

In [15]:
partition, modularity_score = detect_louvain_communities(G, random_state=42)
community_summary_df = summarize_communities(G, labels_dict, partition)

num_communities = community_summary_df["community_id"].nunique()
print(f"Number of communities: {num_communities}")
print(f"Modularity score: {modularity_score:.6f}")

# Community tables
community_summary_path = TABLES_DIR / "community_summary.csv"
community_summary_df.to_csv(community_summary_path, index=False)

top_communities_by_size = community_summary_df.sort_values("size", ascending=False).head(10).copy()
top_by_size_path = TABLES_DIR / "top_communities_by_size.csv"
top_communities_by_size.to_csv(top_by_size_path, index=False)

top_communities_by_ml_ratio = community_summary_df.sort_values(["ml_ratio", "size"], ascending=[False, False]).head(10).copy()
top_by_ml_path = TABLES_DIR / "top_communities_by_ml_ratio.csv"
top_communities_by_ml_ratio.to_csv(top_by_ml_path, index=False)

top_communities_by_web_ratio = community_summary_df.sort_values(["web_ratio", "size"], ascending=[False, False]).head(10).copy()

print(f"Saved: {community_summary_path}")
print(f"Saved: {top_by_size_path}")
print(f"Saved: {top_by_ml_path}")

print("\nTop 10 largest communities:")
display(top_communities_by_size)
print("\nTop communities by ML ratio:")
display(top_communities_by_ml_ratio)
print("\nTop communities by Web ratio:")
display(top_communities_by_web_ratio)

Number of communities: 31
Modularity score: 0.453448
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\community_summary.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\top_communities_by_size.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\top_communities_by_ml_ratio.csv

Top 10 largest communities:


,community_id,size,web_count,ml_count,web_ratio,ml_ratio,dominant_label,purity
0,7,8732,8165,567,0.935066,0.064934,Web,0.935066
1,2,7664,2181,5483,0.284577,0.715423,ML,0.715423
2,13,6454,5205,1249,0.806477,0.193523,Web,0.806477
3,0,3047,2587,460,0.849032,0.150968,Web,0.849032
4,5,2702,2534,168,0.937824,0.062176,Web,0.937824
5,11,2396,1750,646,0.730384,0.269616,Web,0.730384
6,8,2087,1902,185,0.911356,0.088644,Web,0.911356
7,3,2000,1563,437,0.781500,0.218500,Web,0.781500
8,4,904,649,255,0.717920,0.282080,Web,0.717920
9,10,566,472,94,0.833922,0.166078,Web,0.833922



Top communities by ML ratio:


,community_id,size,web_count,ml_count,web_ratio,ml_ratio,dominant_label,purity
18,17,6,0,6,0.000000,1.000000,ML,1.000000
21,23,3,0,3,0.000000,1.000000,ML,1.000000
27,30,3,0,3,0.000000,1.000000,ML,1.000000
15,19,9,1,8,0.111111,0.888889,ML,0.888889
1,2,7664,2181,5483,0.284577,0.715423,ML,0.715423
26,29,3,1,2,0.333333,0.666667,ML,0.666667
13,14,47,26,21,0.553191,0.446809,Web,0.553191
14,15,19,11,8,0.578947,0.421053,Web,0.578947
19,20,5,3,2,0.600000,0.400000,Web,0.600000
16,22,8,5,3,0.625000,0.375000,Web,0.625000



Top communities by Web ratio:


,community_id,size,web_count,ml_count,web_ratio,ml_ratio,dominant_label,purity
20,28,4,4,0,1.000000,0.000000,Web,1.000000
22,18,3,3,0,1.000000,0.000000,Web,1.000000
23,25,3,3,0,1.000000,0.000000,Web,1.000000
24,26,3,3,0,1.000000,0.000000,Web,1.000000
25,27,3,3,0,1.000000,0.000000,Web,1.000000
28,21,3,3,0,1.000000,0.000000,Web,1.000000
29,16,3,3,0,1.000000,0.000000,Web,1.000000
30,1,3,3,0,1.000000,0.000000,Web,1.000000
4,5,2702,2534,168,0.937824,0.062176,Web,0.937824
0,7,8732,8165,567,0.935066,0.064934,Web,0.935066


In [16]:
# Community visualizations
fig4, ax4 = plt.subplots(figsize=(8, 5))
sns.histplot(community_summary_df["size"], bins=40, ax=ax4, color="#4C72B0")
ax4.set_title("Community Size Distribution (Louvain)")
ax4.set_xlabel("Community Size")
ax4.set_ylabel("Count")
fig4.tight_layout()
community_size_dist_path = FIGURES_DIR / "community_size_distribution.png"
fig4.savefig(community_size_dist_path, dpi=300)
plt.close(fig4)

# Top-10 community label composition
top10 = top_communities_by_size.copy()
top10_plot = top10[["community_id", "web_count", "ml_count"]].set_index("community_id")
fig5, ax5 = plt.subplots(figsize=(10, 6))
top10_plot.plot(kind="bar", stacked=True, ax=ax5, color=["#1f77b4", "#ff7f0e"])
ax5.set_title("Top 10 Largest Communities: Web vs ML Composition")
ax5.set_xlabel("Community ID")
ax5.set_ylabel("Number of Nodes")
ax5.legend(["Web", "ML"], title="Label")
fig5.tight_layout()
top_composition_path = FIGURES_DIR / "top_communities_label_composition.png"
fig5.savefig(top_composition_path, dpi=300)
plt.close(fig5)

fig6, ax6 = plt.subplots(figsize=(8, 5))
sns.histplot(community_summary_df["purity"], bins=30, ax=ax6, color="#55A868")
ax6.set_title("Community Purity Distribution")
ax6.set_xlabel("Purity")
ax6.set_ylabel("Count")
fig6.tight_layout()
purity_dist_path = FIGURES_DIR / "community_purity_distribution.png"
fig6.savefig(purity_dist_path, dpi=300)
plt.close(fig6)

sampled_network_path = plot_sampled_network(
    G=G,
    labels_dict=labels_dict,
    partition=partition,
    community_summary_df=community_summary_df,
    output_dir=FIGURES_DIR,
    sample_size=350,
    random_state=42,
)

print("Saved community figures:")
print(f"- {community_size_dist_path}")
print(f"- {top_composition_path}")
print(f"- {purity_dist_path}")
print(f"- {sampled_network_path}")

Saved community figures:
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\community_size_distribution.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\top_communities_label_composition.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\community_purity_distribution.png
- C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\sampled_network_by_label.png


## 10. שיפור החיזוי בעזרת פיצ'רי קהילה
לאחר זיהוי הקהילות, מוסיפים למודל פיצ'רים קהילתיים (כגון מזהה קהילה, גודל קהילה והרכב תוויות ⁦Train⁩ בקהילה) ומריצים מחדש את ניסויי היציבות.

מטרת השלב היא לבדוק האם מידע קהילתי גלובלי משפר את החיזוי מעבר לפיצ'רי גרף מקומיים בלבד.

In [17]:
# Re-run repeated experiments with the community-enhanced model included
repeated_with_comm_df = run_repeated_experiments(
    G=G,
    labels_df=labels_df,
    n_runs=5,
    test_size=0.3,
    random_states=[42, 7, 21, 84, 123],
)

repeated_with_comm_path = TABLES_DIR / "repeated_model_results_with_communities.csv"
repeated_with_comm_df.to_csv(repeated_with_comm_path, index=False)

repeated_with_comm_summary_df = summarize_repeated_results(repeated_with_comm_df)
repeated_with_comm_summary_path = TABLES_DIR / "repeated_model_results_summary_with_communities.csv"
repeated_with_comm_summary_df.to_csv(repeated_with_comm_summary_path, index=False)

print(f"Saved: {repeated_with_comm_path}")
print(f"Saved: {repeated_with_comm_summary_path}")

display(repeated_with_comm_df)
display(repeated_with_comm_summary_df)

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\repeated_model_results_with_communities.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\repeated_model_results_summary_with_communities.csv


,run,random_state,model,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1,roc_auc
0,1,42,Majority Baseline,0.741645,0.500000,0.000000,0.000000,0.000000,0.425830,0.631629,0.500000
1,1,42,Neighbor Majority,0.814147,0.665515,0.822327,0.357974,0.498808,0.692365,0.785909,0.780000
2,1,42,Label Propagation,0.826172,0.688453,0.840942,0.403491,0.545328,0.718937,0.802840,0.789610
3,1,42,Logistic Regression - Graph Features Only,0.799116,0.753619,0.601436,0.659480,0.629122,0.745687,0.802022,0.839559
4,1,42,Logistic Regression - Graph + Community Features,0.814324,0.783720,0.621311,0.720397,0.667195,0.769220,0.818527,0.866639
5,2,7,Majority Baseline,0.741645,0.500000,0.000000,0.000000,0.000000,0.425830,0.631629,0.500000
6,2,7,Neighbor Majority,0.818391,0.676071,0.818649,0.381588,0.520542,0.704261,0.793050,0.789713
7,2,7,Label Propagation,0.828205,0.693280,0.839695,0.414100,0.554664,0.724120,0.806016,0.808894
8,2,7,Logistic Regression - Graph Features Only,0.805393,0.763315,0.611575,0.676249,0.642288,0.754314,0.808454,0.840449
9,2,7,Logistic Regression - Graph + Community Features,0.817153,0.792764,0.622560,0.742300,0.677178,0.774817,0.822005,0.872493


,Model,Accuracy Mean,Accuracy Std,Balanced Accuracy Mean,Balanced Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,Macro F1 Mean,Macro F1 Std,Weighted F1 Mean,Weighted F1 Std,ROC-AUC Mean,ROC-AUC Std
0,Label Propagation,0.828470,0.001761,0.691630,0.002203,0.849505,0.010077,0.408487,0.004032,0.551671,0.004387,0.722809,0.002686,0.805518,0.001894,0.802059,0.009291
1,Logistic Regression - Graph + Community Features,0.816127,0.003347,0.786274,0.004992,0.624225,0.006237,0.724504,0.010672,0.670602,0.006292,0.771534,0.004177,0.820313,0.003242,0.868779,0.005186
2,Logistic Regression - Graph Features Only,0.803307,0.003577,0.758942,0.005337,0.608909,0.005998,0.667146,0.009236,0.636693,0.007292,0.750921,0.004812,0.806126,0.003623,0.840940,0.003707
3,Majority Baseline,0.741645,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.425830,0.000000,0.631629,0.000000,0.500000,0.000000
4,Neighbor Majority,0.818780,0.003222,0.674080,0.005216,0.831198,0.012751,0.374675,0.009899,0.516468,0.010741,0.702481,0.006256,0.792379,0.004110,0.785099,0.005350


In [18]:
# Updated repeated-run plots with all models, including Label Propagation and community features
fig7, ax7 = plt.subplots(figsize=(10, 5))
sns.barplot(data=repeated_with_comm_summary_df, x="Model", y="F1 Mean", palette="Set2", ax=ax7)
ax7.set_title("Repeated Runs (With Community Model): Mean F1")
ax7.set_xlabel("Model")
ax7.set_ylabel("Mean F1")
ax7.tick_params(axis="x", rotation=20)
fig7.tight_layout()
repeated_f1_comm_path = FIGURES_DIR / "repeated_f1_comparison_with_communities.png"
fig7.savefig(repeated_f1_comm_path, dpi=300)
plt.close(fig7)

fig8, ax8 = plt.subplots(figsize=(10, 5))
sns.barplot(data=repeated_with_comm_summary_df, x="Model", y="ROC-AUC Mean", palette="Set3", ax=ax8)
ax8.set_title("Repeated Runs (With Community Model): Mean ROC-AUC")
ax8.set_xlabel("Model")
ax8.set_ylabel("Mean ROC-AUC")
ax8.tick_params(axis="x", rotation=20)
fig8.tight_layout()
repeated_roc_comm_path = FIGURES_DIR / "repeated_roc_auc_comparison_with_communities.png"
fig8.savefig(repeated_roc_comm_path, dpi=300)
plt.close(fig8)

macro_weighted_comm_plot_df = repeated_with_comm_summary_df.melt(
    id_vars=["Model"],
    value_vars=["Macro F1 Mean", "Weighted F1 Mean"],
    var_name="Metric",
    value_name="Score",
)
fig9, ax9 = plt.subplots(figsize=(10, 5))
sns.barplot(data=macro_weighted_comm_plot_df, x="Model", y="Score", hue="Metric", palette="Set1", ax=ax9)
ax9.set_title("Repeated Runs (With Community Model): Macro/Weighted F1")
ax9.set_xlabel("Model")
ax9.set_ylabel("Score")
ax9.tick_params(axis="x", rotation=20)
ax9.legend(title="Metric")
fig9.tight_layout()
repeated_macro_weighted_comm_path = FIGURES_DIR / "repeated_macro_weighted_f1_comparison_with_communities.png"
fig9.savefig(repeated_macro_weighted_comm_path, dpi=300)
plt.close(fig9)

print(f"Saved: {repeated_f1_comm_path}")
print(f"Saved: {repeated_roc_comm_path}")
print(f"Saved: {repeated_macro_weighted_comm_path}")

C:\Users\ofir8\AppData\Local\Temp\ipykernel_34344\3625149064.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=repeated_with_comm_summary_df, x="Model", y="F1 Mean", palette="Set2", ax=ax7)
C:\Users\ofir8\AppData\Local\Temp\ipykernel_34344\3625149064.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=repeated_with_comm_summary_df, x="Model", y="ROC-AUC Mean", palette="Set3", ax=ax8)


Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_f1_comparison_with_communities.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_roc_auc_comparison_with_communities.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_macro_weighted_f1_comparison_with_communities.png


### פרשנות ביניים לשלב הקהילות
מודל Graph+Community משתפר לעומת Graph-Only ולכן מבנה קהילתי מוסיף מידע חיזויי מעבר לשכנות המקומית.

במימוש המעודכן `community_id` אינו נכנס למודל כמספר רגיל, אלא מקודד כקטגוריה באמצעות one-hot encoding. לכן המודל אינו מניח שלמזהי הקהילות יש סדר מספרי מלאכותי.

## 11. כיול סף החלטה
מודל לוגיסטי מחזיר הסתברות למחלקת ⁦ML⁩, וברירת המחדל היא סף 0.5.

בגלל חוסר האיזון, ייתכן שסף אחר ישפר זיהוי של ⁦ML⁩. לכן הסף נבחר על ⁦validation⁩ בלבד (לא על ⁦test⁩), כדי להימנע מ-⁦Test Leakage⁩.

In [19]:
from sklearn.model_selection import train_test_split

# Split original train into inner-train / validation for threshold tuning (no test leakage)
train_inner_nodes, val_nodes, y_train_inner, y_val = train_test_split(
    train_nodes,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

train_inner_nodes = np.asarray(train_inner_nodes, dtype=np.int64)
val_nodes = np.asarray(val_nodes, dtype=np.int64)
y_train_inner = np.asarray(y_train_inner, dtype=np.int64)
y_val = np.asarray(y_val, dtype=np.int64)

nodes_for_threshold = list(train_inner_nodes) + list(val_nodes) + list(test_nodes)

# Build Graph + Community features using only labels from train_inner
# community_id is handled as categorical one-hot in train_logistic_regression_graph_features.
graph_features_threshold_df = extract_graph_neighbor_features(
    G=G,
    labels_dict=labels_dict,
    train_nodes=train_inner_nodes,
    nodes=nodes_for_threshold,
).drop_duplicates(subset=["node_id"])

community_features_threshold_df = add_community_features(
    G=G,
    labels_dict=labels_dict,
    train_nodes=train_inner_nodes,
    nodes=nodes_for_threshold,
    partition=partition,
    community_summary_df=community_summary_df,
).drop_duplicates(subset=["node_id"])

combined_threshold_features_df = (
    graph_features_threshold_df
    .merge(community_features_threshold_df, on="node_id", how="inner")
    .set_index("node_id")
)

X_train_inner = combined_threshold_features_df.reindex(train_inner_nodes).reset_index()
X_val = combined_threshold_features_df.reindex(val_nodes).reset_index()
X_test_threshold = combined_threshold_features_df.reindex(test_nodes).reset_index()

# Train best model type: Logistic Regression - Graph + Community Features
logreg_graph_comm_tuned_model, _, y_score_val = train_logistic_regression_graph_features(
    X_train=X_train_inner,
    y_train=y_train_inner,
    X_test=X_val,
)

# Threshold sweep on validation only
threshold_results_df = evaluate_thresholds(y_true=y_val, y_score=y_score_val)
threshold_results_path = TABLES_DIR / "threshold_tuning_results.csv"
threshold_results_df.to_csv(threshold_results_path, index=False)

best_threshold_f1, best_row_f1 = select_best_threshold(threshold_results_df, metric="f1")
best_threshold_bal_acc, best_row_bal_acc = select_best_threshold(
    threshold_results_df,
    metric="balanced_accuracy",
)

best_thresholds_df = pd.DataFrame([
    {
        "metric_used": "f1",
        "best_threshold": float(best_threshold_f1),
        "accuracy": float(best_row_f1["accuracy"]),
        "balanced_accuracy": float(best_row_f1["balanced_accuracy"]),
        "precision": float(best_row_f1["precision"]),
        "recall": float(best_row_f1["recall"]),
        "f1": float(best_row_f1["f1"]),
        "macro_f1": float(best_row_f1["macro_f1"]),
        "weighted_f1": float(best_row_f1["weighted_f1"]),
        "specificity": float(best_row_f1["specificity"]),
        "tn": int(best_row_f1["tn"]),
        "fp": int(best_row_f1["fp"]),
        "fn": int(best_row_f1["fn"]),
        "tp": int(best_row_f1["tp"]),
    },
    {
        "metric_used": "balanced_accuracy",
        "best_threshold": float(best_threshold_bal_acc),
        "accuracy": float(best_row_bal_acc["accuracy"]),
        "balanced_accuracy": float(best_row_bal_acc["balanced_accuracy"]),
        "precision": float(best_row_bal_acc["precision"]),
        "recall": float(best_row_bal_acc["recall"]),
        "f1": float(best_row_bal_acc["f1"]),
        "macro_f1": float(best_row_bal_acc["macro_f1"]),
        "weighted_f1": float(best_row_bal_acc["weighted_f1"]),
        "specificity": float(best_row_bal_acc["specificity"]),
        "tn": int(best_row_bal_acc["tn"]),
        "fp": int(best_row_bal_acc["fp"]),
        "fn": int(best_row_bal_acc["fn"]),
        "tp": int(best_row_bal_acc["tp"]),
    },
])

best_thresholds_path = TABLES_DIR / "threshold_tuning_best_thresholds.csv"
best_thresholds_df.to_csv(best_thresholds_path, index=False)

# Evaluate chosen thresholds on original test set
y_score_test_threshold = predict_logistic_regression_graph_features(
    logreg_graph_comm_tuned_model,
    X_test_threshold,
)

def _specificity(metrics_dict):
    denom = metrics_dict["tn"] + metrics_dict["fp"]
    return (metrics_dict["tn"] / denom) if denom > 0 else 0.0

# Default threshold = 0.5
y_pred_test_default = (y_score_test_threshold >= 0.5).astype(np.int64)
metrics_test_default = evaluate_predictions(y_test, y_pred_test_default, y_score_test_threshold)

# Best threshold by F1 (selected on validation)
y_pred_test_best_f1 = (y_score_test_threshold >= best_threshold_f1).astype(np.int64)
metrics_test_best_f1 = evaluate_predictions(y_test, y_pred_test_best_f1, y_score_test_threshold)

# Optional: best threshold by balanced accuracy (selected on validation)
y_pred_test_best_bal_acc = (y_score_test_threshold >= best_threshold_bal_acc).astype(np.int64)
metrics_test_best_bal_acc = evaluate_predictions(y_test, y_pred_test_best_bal_acc, y_score_test_threshold)

threshold_test_comparison_df = pd.DataFrame([
    {
        "policy": "default_0.5",
        "threshold": 0.5,
        "accuracy": metrics_test_default["accuracy"],
        "balanced_accuracy": metrics_test_default["balanced_accuracy"],
        "precision": metrics_test_default["precision"],
        "recall": metrics_test_default["recall"],
        "f1": metrics_test_default["f1"],
        "macro_f1": metrics_test_default["macro_f1"],
        "weighted_f1": metrics_test_default["weighted_f1"],
        "specificity": _specificity(metrics_test_default),
        "tn": metrics_test_default["tn"],
        "fp": metrics_test_default["fp"],
        "fn": metrics_test_default["fn"],
        "tp": metrics_test_default["tp"],
    },
    {
        "policy": "best_f1_validation",
        "threshold": float(best_threshold_f1),
        "accuracy": metrics_test_best_f1["accuracy"],
        "balanced_accuracy": metrics_test_best_f1["balanced_accuracy"],
        "precision": metrics_test_best_f1["precision"],
        "recall": metrics_test_best_f1["recall"],
        "f1": metrics_test_best_f1["f1"],
        "macro_f1": metrics_test_best_f1["macro_f1"],
        "weighted_f1": metrics_test_best_f1["weighted_f1"],
        "specificity": _specificity(metrics_test_best_f1),
        "tn": metrics_test_best_f1["tn"],
        "fp": metrics_test_best_f1["fp"],
        "fn": metrics_test_best_f1["fn"],
        "tp": metrics_test_best_f1["tp"],
    },
    {
        "policy": "best_balanced_accuracy_validation",
        "threshold": float(best_threshold_bal_acc),
        "accuracy": metrics_test_best_bal_acc["accuracy"],
        "balanced_accuracy": metrics_test_best_bal_acc["balanced_accuracy"],
        "precision": metrics_test_best_bal_acc["precision"],
        "recall": metrics_test_best_bal_acc["recall"],
        "f1": metrics_test_best_bal_acc["f1"],
        "macro_f1": metrics_test_best_bal_acc["macro_f1"],
        "weighted_f1": metrics_test_best_bal_acc["weighted_f1"],
        "specificity": _specificity(metrics_test_best_bal_acc),
        "tn": metrics_test_best_bal_acc["tn"],
        "fp": metrics_test_best_bal_acc["fp"],
        "fn": metrics_test_best_bal_acc["fn"],
        "tp": metrics_test_best_bal_acc["tp"],
    },
])

threshold_test_comparison_path = TABLES_DIR / "threshold_tuning_test_comparison.csv"
threshold_test_comparison_df.to_csv(threshold_test_comparison_path, index=False)

print(f"Saved: {threshold_results_path}")
print(f"Saved: {best_thresholds_path}")
print(f"Saved: {threshold_test_comparison_path}")
print("\nThreshold comparison on original test set (threshold chosen on validation):")
display(threshold_test_comparison_df)
display(best_thresholds_df)

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\threshold_tuning_results.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\threshold_tuning_best_thresholds.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\threshold_tuning_test_comparison.csv

Threshold comparison on original test set (threshold chosen on validation):


,policy,threshold,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1,specificity,tn,fp,fn,tp
0,default_0.5,0.50,0.812997,0.782380,0.618851,0.719028,0.665189,0.767729,0.817286,0.845732,7094,1294,821,2101
1,best_f1_validation,0.65,0.830504,0.765525,0.687290,0.631075,0.657984,0.772659,0.828081,0.899976,7549,839,1078,1844
2,best_balanced_accuracy_validation,0.45,0.791512,0.781610,0.572606,0.761123,0.653541,0.752216,0.799905,0.802098,6728,1660,698,2224


,metric_used,best_threshold,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1,specificity,tn,fp,fn,tp
0,f1,0.65,0.843880,0.780702,0.718573,0.650037,0.682589,0.789536,0.841246,0.911367,3568,347,477,886
1,balanced_accuracy,0.45,0.802956,0.788506,0.592550,0.758621,0.665380,0.762872,0.810012,0.818391,3204,711,329,1034


In [20]:
# Threshold-tuning plots
fig_thr1, ax_thr1 = plt.subplots(figsize=(8, 5))
ax_thr1.plot(threshold_results_df["threshold"], threshold_results_df["precision"], marker="o", label="Precision")
ax_thr1.plot(threshold_results_df["threshold"], threshold_results_df["recall"], marker="o", label="Recall")
ax_thr1.plot(threshold_results_df["threshold"], threshold_results_df["f1"], marker="o", label="F1")
ax_thr1.set_title("Threshold Tuning (Validation): Precision/Recall/F1")
ax_thr1.set_xlabel("Threshold")
ax_thr1.set_ylabel("Score")
ax_thr1.set_xticks(threshold_results_df["threshold"])
ax_thr1.grid(alpha=0.2)
ax_thr1.legend()
fig_thr1.tight_layout()
threshold_prf_path = FIGURES_DIR / "threshold_f1_recall_precision.png"
fig_thr1.savefig(threshold_prf_path, dpi=300)
plt.close(fig_thr1)

fig_thr2, ax_thr2 = plt.subplots(figsize=(8, 5))
ax_thr2.plot(
    threshold_results_df["threshold"],
    threshold_results_df["balanced_accuracy"],
    marker="o",
    color="#2ca02c",
)
ax_thr2.set_title("Threshold Tuning (Validation): Balanced Accuracy")
ax_thr2.set_xlabel("Threshold")
ax_thr2.set_ylabel("Balanced Accuracy")
ax_thr2.set_xticks(threshold_results_df["threshold"])
ax_thr2.grid(alpha=0.2)
fig_thr2.tight_layout()
threshold_bal_acc_path = FIGURES_DIR / "threshold_balanced_accuracy.png"
fig_thr2.savefig(threshold_bal_acc_path, dpi=300)
plt.close(fig_thr2)

cm_best = confusion_matrix(y_test, y_pred_test_best_f1, labels=[0, 1])
fig_thr3, ax_thr3 = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm_best,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred Web (0)", "Pred ML (1)"],
    yticklabels=["True Web (0)", "True ML (1)"],
    ax=ax_thr3,
)
ax_thr3.set_title(f"Confusion Matrix (Best F1 Threshold={best_threshold_f1:.2f})")
ax_thr3.set_xlabel("Prediction")
ax_thr3.set_ylabel("Truth")
fig_thr3.tight_layout()
cm_best_threshold_path = FIGURES_DIR / "confusion_matrix_best_threshold.png"
fig_thr3.savefig(cm_best_threshold_path, dpi=300)
plt.close(fig_thr3)

print(f"Saved: {threshold_prf_path}")
print(f"Saved: {threshold_bal_acc_path}")
print(f"Saved: {cm_best_threshold_path}")

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\threshold_f1_recall_precision.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\threshold_balanced_accuracy.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\confusion_matrix_best_threshold.png


### פרשנות תוצאות כיול הסף
כיול הסף משנה את האיזון בין ⁦Precision⁩ ל-⁦Recall⁩ במחלקת ⁦ML⁩.

המטרה אינה למקסם ⁦Accuracy⁩ בלבד, אלא לשפר זיהוי של המחלקה הקטנה תוך שמירה על ביצועים כוללים טובים.

In [21]:
delta_f1 = metrics_test_best_f1["f1"] - metrics_test_default["f1"]
delta_recall = metrics_test_best_f1["recall"] - metrics_test_default["recall"]
delta_precision = metrics_test_best_f1["precision"] - metrics_test_default["precision"]
delta_bal_acc = metrics_test_best_f1["balanced_accuracy"] - metrics_test_default["balanced_accuracy"]

print(f"\u05e1\u05e3 \u05de\u05d9\u05d8\u05d1\u05d9 \u05dc\u05e4\u05d9 F1 (\u05e2\u05dc Validation): {best_threshold_f1:.2f}")
print(f"\u05e1\u05e3 \u05de\u05d9\u05d8\u05d1\u05d9 \u05dc\u05e4\u05d9 Balanced Accuracy (\u05e2\u05dc Validation): {best_threshold_bal_acc:.2f}")
print()
print("\u05d4\u05e9\u05d5\u05d5\u05d0\u05d4 \u05e2\u05dc \u05e1\u05d8 \u05d4\u05d1\u05d3\u05d9\u05e7\u05d4 \u05d4\u05de\u05e7\u05d5\u05e8\u05d9:")
print(
    f"- \u05e1\u05e3 0.50: Precision={metrics_test_default['precision']:.4f}, "
    f"Recall={metrics_test_default['recall']:.4f}, F1={metrics_test_default['f1']:.4f}, "
    f"Balanced Accuracy={metrics_test_default['balanced_accuracy']:.4f}"
)
print(
    f"- \u05e1\u05e3 \u05de\u05d9\u05d8\u05d1\u05d9 \u05dc\u05e4\u05d9 F1 ({best_threshold_f1:.2f}): Precision={metrics_test_best_f1['precision']:.4f}, "
    f"Recall={metrics_test_best_f1['recall']:.4f}, F1={metrics_test_best_f1['f1']:.4f}, "
    f"Balanced Accuracy={metrics_test_best_f1['balanced_accuracy']:.4f}"
)
print()
print(
    f"\u05e9\u05d9\u05e0\u05d5\u05d9 \u05dc\u05e2\u05d5\u05de\u05ea \u05e1\u05e3 0.50: \u0394Precision={delta_precision:+.4f}, "
    f"\u0394Recall={delta_recall:+.4f}, \u0394F1={delta_f1:+.4f}, \u0394Balanced Accuracy={delta_bal_acc:+.4f}"
)
print(
    "\u05e4\u05e8\u05e9\u05e0\u05d5\u05ea: \u05db\u05d9\u05d5\u05dc \u05d4\u05e1\u05e3 \u05de\u05e9\u05e0\u05d4 \u05d0\u05ea \u05d4\u05d8\u05e8\u05d9\u05d9\u05d3-\u05d0\u05d5\u05e3 \u05d1\u05d9\u05df Precision \u05dc-Recall \u05d1\u05de\u05d7\u05dc\u05e7\u05ea ML; "
    "\u05d1\u05e4\u05e8\u05d5\u05d9\u05e7\u05d8 \u05d6\u05d4 \u05d6\u05d4 \u05d7\u05e9\u05d5\u05d1 \u05db\u05d9 \u05e4\u05e1\u05e4\u05d5\u05e1 \u05de\u05e4\u05ea\u05d7\u05d9 ML \u05e2\u05dc\u05d5\u05dc \u05dc\u05e4\u05d2\u05d5\u05e2 \u05d1\u05d9\u05db\u05d5\u05dc\u05ea \u05d4\u05d6\u05d9\u05d4\u05d5\u05d9 \u05e9\u05dc \u05d4\u05de\u05d7\u05dc\u05e7\u05d4 \u05d4\u05e7\u05d8\u05e0\u05d4."
)
print("\u05d4\u05de\u05de\u05e6\u05d0\u05d9\u05dd \u05de\u05ea\u05d9\u05d9\u05d7\u05e1\u05d9\u05dd \u05dc\u05e0\u05d9\u05e1\u05d5\u05d9 \u05d6\u05d4 \u05d1\u05dc\u05d1\u05d3 \u05d5\u05d0\u05d9\u05e0\u05dd \u05de\u05e1\u05e7\u05e0\u05d4 \u05e1\u05d5\u05e4\u05d9\u05ea \u05dc\u05db\u05dc \u05d7\u05dc\u05d5\u05e7\u05d4 \u05d0\u05e4\u05e9\u05e8\u05d9\u05ea \u05e9\u05dc \u05d4\u05e0\u05ea\u05d5\u05e0\u05d9\u05dd.")

סף מיטבי לפי F1 (על Validation): 0.65
סף מיטבי לפי Balanced Accuracy (על Validation): 0.45

השוואה על סט הבדיקה המקורי:
- סף 0.50: Precision=0.6189, Recall=0.7190, F1=0.6652, Balanced Accuracy=0.7824
- סף מיטבי לפי F1 (0.65): Precision=0.6873, Recall=0.6311, F1=0.6580, Balanced Accuracy=0.7655

שינוי לעומת סף 0.50: ΔPrecision=+0.0684, ΔRecall=-0.0880, ΔF1=-0.0072, ΔBalanced Accuracy=-0.0169
פרשנות: כיול הסף משנה את הטרייד-אוף בין Precision ל-Recall במחלקת ML; בפרויקט זה זה חשוב כי פספוס מפתחי ML עלול לפגוע ביכולת הזיהוי של המחלקה הקטנה.
הממצאים מתייחסים לניסוי זה בלבד ואינם מסקנה סופית לכל חלוקה אפשרית של הנתונים.


## 12. בדיקת יציבות לכיול הסף
כיול סף על חלוקה אחת אינו מספיק. לכן מריצים כיול סף חוזר על 5 חלוקות, כאשר בכל הרצה:
- בוחרים סף לפי ⁦F1⁩ על ⁦validation⁩ בלבד
- מעריכים על ⁦test⁩ גם בסף 0.5 וגם בסף המכויל

כך מתקבלות תוצאות יציבות יותר ללא דליפת מידע.

In [22]:
repeated_threshold_tuning_results_df = run_repeated_threshold_tuning(
    G=G,
    labels_df=labels_df,
    partition=partition,
    community_summary_df=community_summary_df,
    random_states=[42, 7, 21, 84, 123],
    test_size=0.3,
    validation_size=0.2,
    threshold_metric="f1",
)

repeated_threshold_tuning_results_path = TABLES_DIR / "repeated_threshold_tuning_results.csv"
repeated_threshold_tuning_results_df.to_csv(repeated_threshold_tuning_results_path, index=False)

repeated_threshold_tuning_summary_df = summarize_threshold_tuning_results(
    repeated_threshold_tuning_results_df
)
repeated_threshold_tuning_summary_path = TABLES_DIR / "repeated_threshold_tuning_summary.csv"
repeated_threshold_tuning_summary_df.to_csv(repeated_threshold_tuning_summary_path, index=False)

threshold_stability_comparison_df = (
    repeated_threshold_tuning_results_df
    .groupby("evaluation_type", as_index=False)
    .agg(
        accuracy=("accuracy", "mean"),
        balanced_accuracy=("balanced_accuracy", "mean"),
        precision=("precision", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        macro_f1=("macro_f1", "mean"),
        weighted_f1=("weighted_f1", "mean"),
        roc_auc=("roc_auc", "mean"),
        specificity=("specificity", "mean"),
    )
)

print(f"Saved: {repeated_threshold_tuning_results_path}")
print(f"Saved: {repeated_threshold_tuning_summary_path}")

print("\nתוצאות מלאות לכל ההרצות:")
display(repeated_threshold_tuning_results_df)
print("\nטבלת סיכום (ממוצע ± סטיית תקן):")
display(repeated_threshold_tuning_summary_df)
print("\nהשוואת ממוצעים בין סף ברירת מחדל לסף מכויל:")
display(threshold_stability_comparison_df)

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\repeated_threshold_tuning_results.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\repeated_threshold_tuning_summary.csv

תוצאות מלאות לכל ההרצות:


,run,random_state,selected_threshold,evaluation_type,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1,roc_auc,specificity,tn,fp,fn,tp
0,1,42,0.50,default_threshold_0.5,0.812997,0.782380,0.618851,0.719028,0.665189,0.767729,0.817286,0.862222,0.845732,7094,1294,821,2101
1,1,42,0.65,tuned_threshold_validation_f1,0.830504,0.765525,0.687290,0.631075,0.657984,0.772659,0.828081,0.862222,0.899976,7549,839,1078,1844
2,2,7,0.50,default_threshold_0.5,0.814677,0.788419,0.619226,0.734086,0.671782,0.771335,0.819448,0.869083,0.842752,7069,1319,777,2145
3,2,7,0.55,tuned_threshold_validation_f1,0.823342,0.786901,0.642857,0.711499,0.675439,0.777041,0.826145,0.869083,0.862303,7233,1155,843,2079
4,3,21,0.50,default_threshold_0.5,0.816799,0.782825,0.628244,0.712526,0.667736,0.770635,0.820366,0.861605,0.853124,7156,1232,840,2082
5,3,21,0.60,tuned_threshold_validation_f1,0.830681,0.774900,0.676853,0.659480,0.668053,0.777205,0.829957,0.861605,0.890320,7468,920,995,1927
6,4,84,0.50,default_threshold_0.5,0.816446,0.787715,0.624047,0.728268,0.672142,0.772343,0.820769,0.869924,0.847163,7106,1282,794,2128
7,4,84,0.50,tuned_threshold_validation_f1,0.816446,0.787715,0.624047,0.728268,0.672142,0.772343,0.820769,0.869924,0.847163,7106,1282,794,2128
8,5,123,0.50,default_threshold_0.5,0.810875,0.777827,0.616414,0.709446,0.659666,0.764360,0.814958,0.860068,0.846209,7098,1290,849,2073
9,5,123,0.55,tuned_threshold_validation_f1,0.819540,0.776867,0.640153,0.688569,0.663479,0.770096,0.821624,0.860068,0.865165,7257,1131,910,2012



טבלת סיכום (ממוצע ± סטיית תקן):


,Evaluation Type,Selected Threshold Mean,Selected Threshold Std,Accuracy Mean,Accuracy Std,Balanced Accuracy Mean,Balanced Accuracy Std,Precision Mean,Precision Std,Recall Mean,...,F1 Mean,F1 Std,Macro F1 Mean,Macro F1 Std,Weighted F1 Mean,Weighted F1 Std,ROC-AUC Mean,ROC-AUC Std,Specificity Mean,Specificity Std
0,default_threshold_0.5,0.50,0.000000,0.814359,0.002470,0.783833,0.004339,0.621356,0.004741,0.720671,...,0.667303,0.005159,0.769281,0.003242,0.818565,0.002425,0.86458,0.004572,0.846996,0.003802
1,tuned_threshold_validation_f1,0.57,0.057009,0.824103,0.006408,0.778382,0.009211,0.654240,0.026661,0.683778,...,0.667419,0.006920,0.773869,0.003131,0.825315,0.004006,0.86458,0.004572,0.872985,0.021628



השוואת ממוצעים בין סף ברירת מחדל לסף מכויל:


,evaluation_type,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1,roc_auc,specificity
0,default_threshold_0.5,0.814359,0.783833,0.621356,0.720671,0.667303,0.769281,0.818565,0.86458,0.846996
1,tuned_threshold_validation_f1,0.824103,0.778382,0.654240,0.683778,0.667419,0.773869,0.825315,0.86458,0.872985


In [23]:
# Plot 1: F1 comparison across evaluation types
fig_rt1, ax_rt1 = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=repeated_threshold_tuning_results_df,
    x="evaluation_type",
    y="f1",
    estimator=np.mean,
    ci="sd",
    palette="Set2",
    ax=ax_rt1,
)
ax_rt1.set_title("Repeated Threshold Tuning: F1 Comparison")
ax_rt1.set_xlabel("Evaluation Type")
ax_rt1.set_ylabel("F1")
ax_rt1.tick_params(axis="x", rotation=15)
fig_rt1.tight_layout()
repeated_threshold_f1_path = FIGURES_DIR / "repeated_threshold_f1_comparison.png"
fig_rt1.savefig(repeated_threshold_f1_path, dpi=300)
plt.close(fig_rt1)

# Plot 2: Recall vs Precision comparison
rp_plot_df = repeated_threshold_tuning_results_df.melt(
    id_vars=["run", "evaluation_type"],
    value_vars=["recall", "precision"],
    var_name="metric",
    value_name="value",
)

fig_rt2, ax_rt2 = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=rp_plot_df,
    x="evaluation_type",
    y="value",
    hue="metric",
    estimator=np.mean,
    ci="sd",
    palette="Set1",
    ax=ax_rt2,
)
ax_rt2.set_title("Repeated Threshold Tuning: Recall vs Precision")
ax_rt2.set_xlabel("Evaluation Type")
ax_rt2.set_ylabel("Score")
ax_rt2.tick_params(axis="x", rotation=15)
ax_rt2.legend(title="Metric")
fig_rt2.tight_layout()
repeated_threshold_recall_precision_path = FIGURES_DIR / "repeated_threshold_recall_precision_comparison.png"
fig_rt2.savefig(repeated_threshold_recall_precision_path, dpi=300)
plt.close(fig_rt2)

# Plot 3: selected thresholds per run (plus default line)
selected_thresholds_df = repeated_threshold_tuning_results_df[
    repeated_threshold_tuning_results_df["evaluation_type"] == "tuned_threshold_validation_f1"
][["run", "selected_threshold"]].sort_values("run")

fig_rt3, ax_rt3 = plt.subplots(figsize=(8, 5))
ax_rt3.plot(
    selected_thresholds_df["run"],
    selected_thresholds_df["selected_threshold"],
    marker="o",
    linewidth=2,
    label="Selected threshold (validation F1)",
)
ax_rt3.axhline(0.5, color="red", linestyle="--", label="Default threshold 0.5")
ax_rt3.set_title("Selected Threshold by Run")
ax_rt3.set_xlabel("Run")
ax_rt3.set_ylabel("Threshold")
ax_rt3.set_xticks(selected_thresholds_df["run"].tolist())
ax_rt3.set_ylim(0.0, 1.0)
ax_rt3.grid(alpha=0.2)
ax_rt3.legend()
fig_rt3.tight_layout()
repeated_selected_thresholds_path = FIGURES_DIR / "repeated_selected_thresholds.png"
fig_rt3.savefig(repeated_selected_thresholds_path, dpi=300)
plt.close(fig_rt3)

print(f"Saved: {repeated_threshold_f1_path}")
print(f"Saved: {repeated_threshold_recall_precision_path}")
print(f"Saved: {repeated_selected_thresholds_path}")

C:\Users\ofir8\AppData\Local\Temp\ipykernel_34344\1616684439.py:3: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.barplot(
C:\Users\ofir8\AppData\Local\Temp\ipykernel_34344\1616684439.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
C:\Users\ofir8\AppData\Local\Temp\ipykernel_34344\1616684439.py:30: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar='sd'` for the same effect.

  sns.barplot(


Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_threshold_f1_comparison.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_threshold_recall_precision_comparison.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\repeated_selected_thresholds.png


### פרשנות יציבות לכיול הסף
אם השיפור ב-⁦F1⁩/⁦Recall⁩ חוזר ברוב החלוקות, אפשר להתייחס לכיול הסף כמהלך עקבי ולא כממצא מקרי.

In [24]:
per_run_compare_df = (
    repeated_threshold_tuning_results_df
    .pivot_table(
        index=["run", "random_state"],
        columns="evaluation_type",
        values=["f1", "recall", "precision", "selected_threshold"],
        aggfunc="first",
    )
)

per_run_compare_df.columns = [f"{metric}__{eval_type}" for metric, eval_type in per_run_compare_df.columns]
per_run_compare_df = per_run_compare_df.reset_index()

per_run_compare_df["delta_f1_tuned_minus_default"] = (
    per_run_compare_df["f1__tuned_threshold_validation_f1"]
    - per_run_compare_df["f1__default_threshold_0.5"]
)
per_run_compare_df["delta_recall_tuned_minus_default"] = (
    per_run_compare_df["recall__tuned_threshold_validation_f1"]
    - per_run_compare_df["recall__default_threshold_0.5"]
)
per_run_compare_df["delta_precision_tuned_minus_default"] = (
    per_run_compare_df["precision__tuned_threshold_validation_f1"]
    - per_run_compare_df["precision__default_threshold_0.5"]
)

f1_improved_runs = int((per_run_compare_df["delta_f1_tuned_minus_default"] > 0).sum())
recall_improved_runs = int((per_run_compare_df["delta_recall_tuned_minus_default"] > 0).sum())

default_means = threshold_stability_comparison_df[
    threshold_stability_comparison_df["evaluation_type"] == "default_threshold_0.5"
].iloc[0]
tuned_means = threshold_stability_comparison_df[
    threshold_stability_comparison_df["evaluation_type"] == "tuned_threshold_validation_f1"
].iloc[0]

threshold_stats = repeated_threshold_tuning_summary_df[
    repeated_threshold_tuning_summary_df["Evaluation Type"] == "tuned_threshold_validation_f1"
].iloc[0]

print(f"סף מכויל ממוצע שנבחר: {threshold_stats['Selected Threshold Mean']:.4f} ± {threshold_stats['Selected Threshold Std']:.4f}")
print()
print("ממוצעים על 5 החלוקות:")
print(
    f"- סף 0.5: Precision={default_means['precision']:.4f}, Recall={default_means['recall']:.4f}, "
    f"F1={default_means['f1']:.4f}, Balanced Accuracy={default_means['balanced_accuracy']:.4f}"
)
print(
    f"- סף מכויל: Precision={tuned_means['precision']:.4f}, Recall={tuned_means['recall']:.4f}, "
    f"F1={tuned_means['f1']:.4f}, Balanced Accuracy={tuned_means['balanced_accuracy']:.4f}"
)
print()
print(
    f"שינוי ממוצע (מכויל פחות ברירת מחדל): "
    f"ΔPrecision={tuned_means['precision'] - default_means['precision']:+.4f}, "
    f"ΔRecall={tuned_means['recall'] - default_means['recall']:+.4f}, "
    f"ΔF1={tuned_means['f1'] - default_means['f1']:+.4f}"
)
print(f"F1 השתפר ב-{f1_improved_runs}/5 חלוקות")
print(f"Recall השתפר ב-{recall_improved_runs}/5 חלוקות")
print("הפרשנות נשארת זהירה: מדובר במגמה אמפירית על 5 חלוקות, לא במסקנה סופית לכל חלוקה אפשרית.")

print("\nהשוואה לפי חלוקה:")
display(per_run_compare_df[[
    "run",
    "random_state",
    "selected_threshold__tuned_threshold_validation_f1",
    "delta_f1_tuned_minus_default",
    "delta_recall_tuned_minus_default",
    "delta_precision_tuned_minus_default",
]])

סף מכויל ממוצע שנבחר: 0.5700 ± 0.0570

ממוצעים על 5 החלוקות:
- סף 0.5: Precision=0.6214, Recall=0.7207, F1=0.6673, Balanced Accuracy=0.7838
- סף מכויל: Precision=0.6542, Recall=0.6838, F1=0.6674, Balanced Accuracy=0.7784

שינוי ממוצע (מכויל פחות ברירת מחדל): ΔPrecision=+0.0329, ΔRecall=-0.0369, ΔF1=+0.0001
F1 השתפר ב-3/5 חלוקות
Recall השתפר ב-0/5 חלוקות
הפרשנות נשארת זהירה: מדובר במגמה אמפירית על 5 חלוקות, לא במסקנה סופית לכל חלוקה אפשרית.

השוואה לפי חלוקה:


,run,random_state,selected_threshold__tuned_threshold_validation_f1,delta_f1_tuned_minus_default,delta_recall_tuned_minus_default,delta_precision_tuned_minus_default
0,1,42,0.65,-0.007205,-0.087953,0.068439
1,2,7,0.55,0.003657,-0.022587,0.023631
2,3,21,0.60,0.000318,-0.053046,0.048609
3,4,84,0.50,0.000000,0.000000,0.000000
4,5,123,0.55,0.003813,-0.020876,0.023739


## 13. ניתוח טעויות מתקדם
סעיף זה בוחן איפה המודל המשופר נכשל: לפי דרגת הצומת, לפי טוהר הקהילה, ולפי כיוון הטעות. הניתוח מבוצע לאחר ההערכה ואינו משמש לאימון המודל.

In [25]:
# advanced_error_analysis_marker
# Error analysis for the tuned Graph + Community model
error_by_degree_tuned_df = evaluate_by_degree_group(
    G=G,
    test_nodes=test_nodes,
    y_true=y_test,
    y_pred=y_pred_test_best_f1,
)
error_by_degree_tuned_path = TABLES_DIR / "error_analysis_by_degree_group_tuned_model.csv"
error_by_degree_tuned_df.to_csv(error_by_degree_tuned_path, index=False)

community_purity_error_df = evaluate_errors_by_community_purity(
    test_nodes=test_nodes,
    y_true=y_test,
    y_pred=y_pred_test_best_f1,
    partition=partition,
    community_summary_df=community_summary_df,
)
community_purity_error_path = TABLES_DIR / "community_purity_error_analysis.csv"
community_purity_error_df.to_csv(community_purity_error_path, index=False)

error_types_df = summarize_error_types(y_test, y_pred_test_best_f1)
error_types_path = TABLES_DIR / "prediction_error_types.csv"
error_types_df.to_csv(error_types_path, index=False)

# Degree error-rate plot for tuned model
plot_degree_tuned_df = error_by_degree_tuned_df.melt(
    id_vars=["degree_group", "count"],
    value_vars=["accuracy", "balanced_accuracy", "f1", "macro_f1", "weighted_f1"],
    var_name="metric",
    value_name="value",
)
fig_err_deg, ax_err_deg = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_degree_tuned_df, x="degree_group", y="value", hue="metric", ax=ax_err_deg)
ax_err_deg.set_title("Tuned Model Error Analysis by Degree Group")
ax_err_deg.set_xlabel("Degree Group")
ax_err_deg.set_ylabel("Score")
ax_err_deg.tick_params(axis="x", rotation=15)
ax_err_deg.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
fig_err_deg.tight_layout()
error_degree_tuned_plot_path = FIGURES_DIR / "error_analysis_by_degree_group_tuned_model.png"
fig_err_deg.savefig(error_degree_tuned_plot_path, dpi=300)
plt.close(fig_err_deg)

community_purity_error_plot_path = plot_community_purity_error_rates(community_purity_error_df, FIGURES_DIR)
error_types_plot_path = plot_error_type_counts(error_types_df, FIGURES_DIR)

print(f"Saved: {error_by_degree_tuned_path}")
print(f"Saved: {community_purity_error_path}")
print(f"Saved: {error_types_path}")
print(f"Saved: {error_degree_tuned_plot_path}")
print(f"Saved: {community_purity_error_plot_path}")
print(f"Saved: {error_types_plot_path}")

print("\nError analysis by degree group:")
display(error_by_degree_tuned_df)
print("\nError analysis by community purity:")
display(community_purity_error_df)
print("\nPrediction outcome types:")
display(error_types_df)

Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\error_analysis_by_degree_group_tuned_model.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\community_purity_error_analysis.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\tables\prediction_error_types.csv
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\error_analysis_by_degree_group_tuned_model.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\community_purity_error_analysis.png
Saved: C:\Dev\Graph-Algo-Course\github-social-network\outputs\figures\prediction_error_types.png

Error analysis by degree group:


C:\Dev\Graph-Algo-Course\github-social-network\src\visualization.py:203: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=error_types_df, x="error_type", y="count", palette="Set2", ax=ax)


,degree_group,count,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1
0,degree_0_2,2877,0.763643,0.723740,0.687500,0.591633,0.635974,0.730494,0.759043
1,degree_3_5,2784,0.818606,0.770922,0.707510,0.654878,0.680177,0.776790,0.816489
2,degree_6_10,2207,0.848663,0.784998,0.714567,0.657609,0.684906,0.792662,0.846515
3,degree_11_30,2367,0.870722,0.779786,0.648585,0.636574,0.642523,0.781808,0.870252
4,degree_31_plus,1075,0.914419,0.801372,0.585938,0.657895,0.619835,0.785808,0.916580



Error analysis by community purity:


,purity_group,count,errors,error_rate
0,mixed_lt_0.70,35,15,0.428571
1,medium_0.70_0.85,6951,1584,0.227881
2,pure_0.85_plus,4324,318,0.073543



Prediction outcome types:


,error_type,count,ratio
0,Correct Web,7549,0.667462
1,Correct ML,1844,0.163042
2,Web -> ML,839,0.074182
3,ML -> Web,1078,0.095314


## 14. דיון בתוצאות
- **⁦Majority Baseline⁩** נכשל בזיהוי ⁦ML⁩, כצפוי בדאטה לא-מאוזן.
- **⁦Neighbor Majority⁩** מראה שקשרי שכנות מכילים מידע שימושי.
- **⁦Label Propagation⁩** מוסיף אלגוריתם גרפי ישיר שמפיץ תוויות מ-train דרך הקשתות בלי להשתמש בתוויות test.
- **⁦Logistic Regression⁩ - ⁦Graph Features Only⁩** משפר ביצועים בעזרת מאפיינים מבניים כמו PageRank, clustering ו-core number.
- **⁦Logistic Regression⁩ - ⁦Graph + Community Features⁩** נותן את התוצאה הטובה ביותר ומראה שמבנה קהילתי גלובלי מוסיף סיגנל חיזויי.

תוצאות חוזרות (5 חלוקות, ממוצעים):

⁦Majority Baseline⁩: Accuracy 0.7416 | Balanced Accuracy 0.5000 | F1 0.0000 | Macro F1 0.4258 | Weighted F1 0.6316 | ROC-AUC 0.5000

⁦Neighbor Majority⁩: Accuracy 0.8188 | Balanced Accuracy 0.6741 | F1 0.5165 | Macro F1 0.7025 | Weighted F1 0.7924 | ROC-AUC 0.7851

⁦Label Propagation⁩: Accuracy 0.8285 | Balanced Accuracy 0.6916 | F1 0.5517 | Macro F1 0.7228 | Weighted F1 0.8055 | ROC-AUC 0.8021

⁦Logistic Regression⁩ - ⁦Graph Features Only⁩: Accuracy 0.8033 | Balanced Accuracy 0.7589 | F1 0.6367 | Macro F1 0.7509 | Weighted F1 0.8061 | ROC-AUC 0.8409

⁦Logistic Regression⁩ - ⁦Graph + Community Features⁩: Accuracy 0.8161 | Balanced Accuracy 0.7862 | F1 0.6706 | Macro F1 0.7715 | Weighted F1 0.8203 | ROC-AUC 0.8687

כיול סף:
בזכות הוספת משקולות (class_weight="balanced") למודל הלוגיסטי, המודל התאזן באופן טבעי. סף ברירת המחדל (0.50) כבר סיפק Recall מעולה של 0.720 ו-F1 של 0.667. כיול הסף בחר להעלות את הרף ל-0.57 (בממוצע), מה ששיפר את ה-Precision ל-0.654 תוך שמירה על Recall יציב של 0.683, וכך מקסם את יעילות הזיהוי של מחלקת המיעוט.

## 15. מגבלות העבודה
- הדאטה הוא מיוני 2019 ולכן אינו מייצג בהכרח את ⁦GitHub⁩ כיום.
- המשימה בינארית בלבד: ⁦Web⁩ מול ⁦Machine Learning⁩, אף שבפועל קיימים תחומי התמחות רבים וחופפים.
- התוויות נגזרו מתיאורי תפקידים ועלולות לכלול רעש.
- הגרף סטטי ואינו כולל התפתחות בזמן.
- במכוון לא השתמשנו בפיצ'רי צומת גולמיים, ולכן כוח החיזוי מוגבל למידע מבני-רשתי.

## 16. מסקנות
- ניתן לחזות את תחום ההתמחות של מפתח ⁦GitHub⁩ במידה משמעותית באמצעות קשרי מעקב הדדיים בלבד.
- הפרויקט נשאר ⁦network-only⁩: אין שימוש בתוכן טקסטואלי, פרופילים או מאפייני repositories.
- הרשת מציגה הומופיליה חזקה, ולכן שכנות וקהילות מספקות מידע חיזויי.
- ⁦Label Propagation⁩ מחזק את הצד האלגוריתמי-גרפי של העבודה ומראה שתוויות יכולות להתפשט דרך מבנה הקשרים.
- מאפייני גרף כמו PageRank, clustering ו-core number משפרים את החיזוי מעבר למודל שכנות פשוט.
- מבנה קהילתי מסוג ⁦Louvain⁩ משפר את החיזוי מעבר לפיצ'רי גרף מקומיים.
- המודל הטוב ביותר כברירת מחדל הוא ⁦Logistic Regression⁩ - ⁦Graph + Community Features⁩.
- התשובה לשאלת המחקר חיובית אך לא מושלמת: יש סיגנל חיזויי חזק, אך עדיין קיימות טעויות בעיקר בקהילות מעורבות ובזיהוי חלק ממפתחי ⁦ML⁩.

## 17. מקורות
- ⁦MUSAE GitHub Social Network⁩ dataset.
- ⁦Multi-scale Attributed Node Embedding⁩ (לפי `data/citing.txt`).
- Blondel et al., Fast unfolding of communities in large networks.
- PageRank literature and course materials.
- חומרי הקורס: centrality, community analysis, network models, link prediction, ומבוא ל-⁦Graph Neural Networks⁩.